# 02 · The last connections of the night

What are the last connections from Stuttgart, Freiburg and Konstanz back to Villingen-area towns and villages on a Friday, and how likely is each one to actually get you home? (Deutschlandticket, delay model + simulator.)

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
ROOT = Path.cwd().resolve().parents[1]
sys.path.insert(0, str(ROOT / "backend"))
import numpy as np, pandas as pd
pd.set_option("display.width", 140)

In [2]:
from datetime import date
from app.services.engine_state import get_engine
from app.engine.raptor import seconds_to_hhmm as hhmm
eng = get_engine()
planner, store = eng.planner, eng.store
day = date(2026, 9, 25)  # a Friday
print("model", eng.model.version)

model v1


In [3]:
origins = ["Stuttgart Hauptbahnhof (oben)", "Freiburg Hauptbahnhof", "Konstanz Bahnhof"]
destinations = ["Villingen Bahnhof/ZOB", "Schwenningen Bahnhof", "Donaueschingen Bahnhof", "St. Georgen Bahnhof",
                "Triberg Bahnhof", "Bad Dürrheim ZOB", "Königsfeld ZOB", "Schonach Rathaus", "Vöhrenbach ZOB",
                "Bräunlingen Bahnhof"]
rows = []
for o in origins:
    origin = store.resolve_station(o)
    for d in destinations:
        dest = store.resolve_station(d)
        tail = planner.last_connections(day, origin, dest, regional_only=True)
        if not tail:
            rows.append({"from": origin.name, "to": dest.name}); continue
        last = planner.evaluate(tail[-1], dest, True, {}, origin, later_from_origin=tail)
        safe = None
        for journey in reversed(tail):
            ev = last if journey is tail[-1] else planner.evaluate(journey, dest, True, {}, origin,
                                                                  later_from_origin=tail)
            if ev.p_home >= 0.95:
                safe = ev; break
        rows.append({"from": origin.name.split(" ")[0], "to": dest.name,
                     "last departure": hhmm(last.journey.departure), "arrives": hhmm(last.journey.arrival),
                     "changes": last.journey.transfers, "P(home) last": round(last.p_home, 2),
                     "latest ≥95% safe": hhmm(safe.journey.departure) if safe else "none"})
table = pd.DataFrame(rows)
table

,from,to,last departure,arrives,changes,P(home) last,latest ≥95% safe
0,Stuttgart,Villingen Bahnhof/ZOB,22:17,00:28,1,0.81,21:14
1,Stuttgart,Schwenningen Bahnhof,22:17,00:16,1,0.83,21:14
2,Stuttgart,Donaueschingen Bahnhof,20:23,22:37,1,0.79,19:14
3,Stuttgart,St. Georgen Bahnhof,20:59,00:27,2,0.72,19:32
4,Stuttgart,Triberg Bahnhof,20:59,00:12,2,0.73,19:32
5,Stuttgart,Bad Dürrheim ZOB,20:23,22:24,2,0.18,none
6,Stuttgart,Königsfeld ZOB,20:23,23:19,2,0.33,none
7,Stuttgart,Schonach Rathaus,15:14,18:34,3,0.23,none
8,Stuttgart,Vöhrenbach ZOB,21:14,00:02,2,0.51,none
9,Stuttgart,Bräunlingen Bahnhof,20:23,22:50,1,0.36,19:14


**Reading the table:** the last train on the timetable is often *not* the one to plan on. When the last connection needs a tight change, its chance of getting you home is noticeably lower, and the latest departure that is at least 95% safe is often an hour earlier.

In [4]:
print(table.to_markdown(index=False))

| from      | to                     | last departure   | arrives   |   changes |   P(home) last | latest ≥95% safe   |
|:----------|:-----------------------|:-----------------|:----------|----------:|---------------:|:-------------------|
| Stuttgart | Villingen Bahnhof/ZOB  | 22:17            | 00:28     |         1 |           0.81 | 21:14              |
| Stuttgart | Schwenningen Bahnhof   | 22:17            | 00:16     |         1 |           0.83 | 21:14              |
| Stuttgart | Donaueschingen Bahnhof | 20:23            | 22:37     |         1 |           0.79 | 19:14              |
| Stuttgart | St. Georgen Bahnhof    | 20:59            | 00:27     |         2 |           0.72 | 19:32              |
| Stuttgart | Triberg Bahnhof        | 20:59            | 00:12     |         2 |           0.73 | 19:32              |
| Stuttgart | Bad Dürrheim ZOB       | 20:23            | 22:24     |         2 |           0.18 | none               |
| Stuttgart | Königsfeld ZOB         | 2